# From 2D to 1D: circular, sector and annular averages

A detector image is often most useful as a few 1D curves. IQuana makes three:

| Call | Gives | Typical use |
|---|---|---|
| `circular_average()` | I(Q) over the whole detector | compare with the reduced 1D file |
| `sector_average(phi)` | I(Q) inside a sector | along and across a field or shear direction |
| `annular_average(qmin, qmax)` | I(φ) inside a ring | anisotropy, the sin² law of magnetic SANS |

Each is a method of the fitter, which already knows the mask, the Q range and
the resolution, and a function in `data_ops` for a bare dataset. The two I(Q)
results are ordinary fit-ready `Data1D` objects.

This notebook walks through:

1. A circular average, checked against the facility's own 1D reduction
2. Sectors along and across the field of a polarised (SANSPOL) image
3. `plot_cuts()`: did I cut where I meant to?
4. The anisotropy around a ring, with `fit_sin2()`
5. Combining cuts of several files on one Q grid
6. Fitting the cuts

In [ ]:
import warnings

import numpy as np
import plotly.graph_objects as go

from iquana import IQuana, MultiFitter, data_ops, examples, modelfree, set_verbosity

set_verbosity('info')   # the averages report what they did in one line

## 1. A circular average

`sans2d_2d` is a SANS2D detector image; `canSAS_xml` is the 1D reduction of the
same run. The preset fitter has the beam stop and the zero-dI pixels masked, and
the cut uses exactly the pixels a 2D fit would use.

With no argument the bins are as wide as the pixel pitch, starting at Q = 0, so
the grid depends only on the image.

In [ ]:
sans2d = examples.load_fitter('sans2d_2d')
iq = sans2d.circular_average()

`bins=` also takes a curve. Given the reduced 1D file, the cut lands on that
file's Q points, so the two can be compared point for point.

In [ ]:
reduced = examples.load('canSAS_xml')
on_file_grid = sans2d.circular_average(bins=reduced)

ratio = on_file_grid.y / reduced.y
fig = go.Figure(go.Scatter(x=reduced.x, y=ratio, mode='markers'))
fig.update_layout(
    title='Circular average of the image / reduced 1D file',
    xaxis={'title': 'Q (Å⁻¹)', 'type': 'log'},
    yaxis={'title': 'ratio', 'range': [0.5, 1.5]},
    template='plotly_white', height=380,
)
fig

Between Q ≈ 0.03 and 0.11 Å⁻¹ they agree within a few percent, mostly 1 %.
Below that, a bin of
this image holds only a handful of pixels (`on_file_grid.averaging['n_pixels']`)
and scatters much more.

Empty bins and bins that the beam stop cuts through stay in the curve as NaN.
IQuana masks NaN points everywhere, so they are neither fitted nor plotted:

In [ ]:
print(f'{iq.x.size} bins, {np.isfinite(iq.y).sum()} with data, '
      f"{iq.averaging['edge_bins'].size} edge bin(s) dropped")

## 2. Sectors along and across the field

`sanspol_plus` is a SANSPOL image of magnetic nanoparticles in a horizontal
field (spin-up incident beam). Angles are in degrees, anticlockwise from +Qx;
`'horizontal'` and `'vertical'` are 0° and 90°. By default a sector is ±10° wide
and includes the opposite side of the beam (`mirror=True`).

Which detector direction is the field depends on the instrument; here it is
horizontal.

In [ ]:
plus = examples.load_fitter('sanspol_plus')
along = plus.sector_average('horizontal')
across = plus.sector_average('vertical')

The S50 image merges two detector distances, so it has two pixel sizes. The log
line says so, and the bins follow: 0.00104 Å⁻¹ wide at low Q, about 0.0039 above
Q ≈ 0.1.

On I = A + B·sin²θ a ±10° sector mixes in 1 % of the other direction; the
docstring of `sector_average` has the table for other widths.

## 3. Did I cut where I meant to?

In [ ]:
ring_plus = plus.annular_average(0.02, 0.04)
plus.plot_cuts(along, across, ring_plus)

## 4. The anisotropy around a ring

`fit_sin2()` fits I(φ) = A + B·sin²(φ − φ₀): the law of a saturated magnetic
sample with the field in the detector plane, and the lowest harmonic of any
twofold pattern.

With φ₀ free, B comes out positive and φ₀ is **the direction of least
intensity**. That is the field direction only where the physics makes B
positive.

In [ ]:
fit_plus = ring_plus.fit_sin2()
print(fit_plus)
fit_plus.plot()

In the spin-down channel the intensity is lowest *across* the field, so a free
fit points 90° away. Fix φ₀ to the direction found in I⁺: B then keeps its sign
and comes out negative.

In [ ]:
minus = examples.load_fitter('sanspol_minus')
ring_minus = minus.annular_average(0.02, 0.04)
print('free fit, phi0 =', round(ring_minus.fit_sin2().phi0, 1))
fixed = ring_minus.fit_sin2(phi0=fit_plus.phi0)
print(f'phi0 fixed: B = {fixed.b:.3f} +/- {fixed.b_err:.3f}')

When the anisotropy is too weak to fix a direction (B less than 3 σ), the fit
warns `weak_anisotropy` and gives φ₀ no error bar.

## 5. Cuts of several files on one grid

All cuts of **one** image share one Q grid, so `data_ops.subtract(across, along)`
works at once. In one spin channel that difference is the anisotropic part of
the channel, |M|² ∓ 2·Re(N M*), not the magnetic scattering alone.

Two files have their own pixel coordinates, so their default grids need not
agree. `bins=<a cut>` puts a cut of the second file on the grid of the first.

In [ ]:
unpol = examples.load_fitter('sanspol_unpol')

anisotropic = data_ops.subtract(across, along)            # one channel
magnetic = data_ops.subtract(                             # unpolarised: the magnetic term
    unpol.sector_average('vertical', bins=across),
    unpol.sector_average('horizontal', bins=across),
)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')                       # NaN rows of empty bins
    cross_term = data_ops.subtract(minus.sector_average('vertical', bins=across), across)

fig = go.Figure()
for name, curve in (('across − along, I⁺', anisotropic),
                    ('across − along, unpolarised', magnetic),
                    ('I⁻ − I⁺, across', cross_term)):
    fig.add_trace(go.Scatter(x=curve.x, y=curve.y, mode='markers', name=name))
fig.update_layout(xaxis={'title': 'Q (Å⁻¹)', 'type': 'log', 'range': [-2.1, -0.9]},
                  yaxis={'title': 'ΔI'}, template='plotly_white', height=420)
fig

Subtracting two cuts that share pixels (a circular average minus a sector, two
overlapping sectors) gives a wrong dI, because the uncertainties are propagated
as if the curves were independent. `data_ops.subtract` warns `shared_pixels`
in that case.

## 6. Fitting the cuts

A cut is an ordinary 1D dataset: `set_data()`, `MultiFitter.add()`,
`fit_series()`, `modelfree` and P(r) all take it. Here both sectors share one
radius in a joint fit, and the unpolarised circular average gives a Guinier
radius.

In [ ]:
set_verbosity('quiet')
joint = MultiFitter()
for name, cut in (('along', along), ('across', across)):
    joint.add(name, cut, model='sphere')
    joint[name].set_param('radius', value=70, min=10, max=200, vary=True)
    joint[name].set_param('scale', value=0.002, min=1e-6, max=1, vary=True)
    joint[name].set_param('background', value=0.1, min=0, max=5, vary=True)
    joint[name].set_resolution('none')
joint.share('radius')
result = joint.fit(method='lm')
joint.plot_results()

In [ ]:
guinier = modelfree.guinier(unpol.circular_average(), qmax=0.019)
print(guinier)

## Things to know

- **Resolution.** The fitter methods carry the fitter's resolution setting
  over: the file's radial width with `'data'`, `dq_over_q·|Q|` per pixel with
  `'pinhole'`, plus the spread of |Q| inside each bin. None of the bundled 2D
  files has dQ, so `set_resolution('pinhole', dq_over_q=...)` on the 2D fitter
  is the usual route.
- **Pixels with dI = 0** are kept, as the fitter keeps them, and reported. If
  they are dead pixels, call `fitter.mask_invalid()` first.
- **Without dI** the uncertainty of a bin is the scatter of its pixels, which
  overestimates it, and bins with fewer than 5 pixels are left out.
- **Isotropy.** P(r), the invariant and Porod assume isotropic scattering. On a
  sector of anisotropic data they run, but the numbers are not what their names
  promise.
- **SasView gives different numbers** in some cases: see "Differences from
  SasView" in the user guide.